# ✈️ Orquestación del ETL OpenSky con Prefect  
## Ejecución del flujo `etl_opensky_flow` y scheduling opcional

Este notebook acompaña al pipeline principal de **ETL de Tráfico Aéreo con OpenSky Network**, ya implementado siguiendo la arquitectura **Bronze → Silver → Gold** y documentado en `01_opensky_etl.ipynb`.

Mientras que el notebook anterior se centra en la **ejecución manual paso a paso** (ingesta, limpieza, enriquecimiento y visualizaciones), aquí el foco está en la **orquestación con Prefect**, utilizando la lógica definida en:

- `src/etl_utils.py` → funciones auxiliares de extracción, transformación y guardado  
- `src/etl_opensky_flow.py` → definición del flujo `etl_opensky_flow` (tasks + flow Prefect)

El flujo automatiza el recorrido completo:

- 📥 **Extracción** del snapshot dinámico desde la API pública de OpenSky  
- 🟤 **Bronze** → normalización básica y persistencia cruda en Delta Lake  
- 🥈 **Silver** → limpieza, tipificación, columnas temporales y particionado por hora  
- 🟡 **Gold** → lectura desde Silver, enriquecimiento con metadatos estáticos y guardado final  

---

## 🎯 Objetivo de este notebook

Este notebook está pensado para:

- ejecutar el flujo **`etl_opensky_flow` de forma manual**, desde Prefect  
- verificar que las tareas de cada capa se encadenan correctamente  
- revisar logs de ejecución y comportamiento general del pipeline  
- documentar una posible **ejecución programada** (cron) sin activarla por defecto

No se redefinen transformaciones ni lógica de negocio:  
simplemente se **importa el flujo ya implementado** y se lo ejecuta en un contexto controlado.

---

## 📘 Estructura de este notebook

1. **Configuración mínima e importación del flujo**
2. **Ejecución manual del pipeline `etl_opensky_flow()`**
3. **(Opcional, documentado) Ejecución programada con `serve` y cron**

Este notebook funciona como interfaz de orquestación y validación, complementando al notebook principal de ETL y preparando el proyecto para futuras integraciones con **Prefect Cloud** o despliegues en **Azure**.

## Uso del flujo definido en `src/etl_opensky_flow.py`

El flujo ETL está implementado en `src/etl_opensky_flow.py` y encapsula el pipeline completo:

- extracción desde OpenSky  
- normalización y guardado en Bronze  
- limpieza, tipificación y particionado en Silver  
- enriquecimiento y persistencia final en Gold  

Desde este notebook se puede:

- **Opción A — correr una ejecución única (demo one-off)** para validar la orquestación  
- **Opción B — servir el flow (opcional)** para mantenerlo activo con ejecución programada


In [1]:
import prefect
print("Prefect versión:", prefect.__version__)

Prefect versión: 2.20.9


### Opción A — Corrida manual del flujo (one-off)

Esta modalidad ejecuta el flujo `etl_opensky_flow` **una sola vez**, ideal para validar
que todas las tareas del pipeline funcionan correctamente:

- extracción del snapshot dinámico  
- limpieza y particionado en Silver  
- enriquecimiento con metadatos estáticos  
- guardado final en Gold  

Se recomienda esta opción para pruebas, validación local y depuración.

### Opción A — Corrida manual del flujo (one-off)

Ejecuta el flujo `etl_opensky_flow` **una sola vez**, ideal para validar que la orquestación funciona correctamente. Permite verificar:

- que la extracción desde OpenSky responde  
- que las transformaciones Bronze → Silver → Gold se encadenan sin errores  
- que el flujo persiste los datos en el Data Lake como se espera  

Esta modalidad es la recomendada para pruebas locales y depuración antes de activar cualquier programación automática.


In [2]:
import sys
import os

# Se agrega la carpeta src al path (sube un nivel desde /notebooks)
sys.path.append(os.path.abspath("../src"))

In [3]:
import importlib

# Importa el módulo de orquestación
etl = importlib.import_module("etl_opensky_flow")

# Ejecuta el flujo ETL de forma local (modo recomendado dentro del Notebook)
etl.etl_opensky_flow()

# Nota:
# También puede ejecutarse desde la terminal:
#     python src/etl_opensky_flow.py
#
# O desde el notebook:
#     !python ../src/etl_opensky_flow.py
#
# Todas las opciones ejecutan exactamente el mismo flow.

10:32:41.773 | INFO    | prefect.engine - Created flow run 'silent-toucan' for flow 'etl-opensky-full-pipeline'

10:32:41.791 | INFO    | Flow run 'silent-toucan' - View at https://app.prefect.cloud/account/1513bf29-3686-40b8-9dbf-c85ba6a6f8c0/workspace/377710aa-6343-48a1-a52d-8fd9be6fbba7/flow-runs/flow-run/0692d98f-9a04-7064-8000-b0d5b0e3bde0

10:32:42.537 | INFO    | Flow run 'silent-toucan' - Created task run 'task_extract_aircraft_metadata-0' for task 'task_extract_aircraft_metadata'

10:32:42.537 | INFO    | Flow run 'silent-toucan' - Executing 'task_extract_aircraft_metadata-0' immediately...

10:33:30.573 | INFO    | Task run 'extract-aircraft-metadata' - Finished in state Completed()

...

💾 Datos guardados en Delta Lake: c:\Users\elias\OneDrive\Documentos\Repositorios_notebooks\ETL_OpenSky_Aviation\data\etl_datalake\gold\api_opensky


10:33:56.749 | INFO    | Task run 'save-gold-states' - Finished in state Completed()

✅ ETL OpenSky ejecutado correctamente.


10:33:57.229 | INFO    | Flow run 'silent-toucan' - Finished in state Completed('All states completed.')

[Completed(message=None, type=COMPLETED, result=UnpersistedResult(type='unpersisted', artifact_type='result', artifact_description='Unpersisted result of type `DataFrame`')),
 Completed(message=None, type=COMPLETED, result=UnpersistedResult(type='unpersisted', artifact_type='result', artifact_description='Unpersisted result of type `bool`')),
 Completed(message=None, type=COMPLETED, result=UnpersistedResult(type='unpersisted', artifact_type='result', artifact_description='Unpersisted result of type `DataFrame`')),
 Completed(message=None, type=COMPLETED, result=UnpersistedResult(type='unpersisted', artifact_type='result', artifact_description='Unpersisted result of type `bool`')),
 Completed(message=None, type=COMPLETED, result=UnpersistedResult(type='unpersisted', artifact_type='result', artifact_description='Unpersisted result of type `dict`')),
 Completed(message=None, type=COMPLETED, result=UnpersistedResult(type='unpersisted', artifact_type='result', artifact_description='Unpersis

### Opción B — Servir el flow como agente local (opcional)

Esta modalidad permite ejecutar el flujo de manera **continua**, manteniéndolo activo como un servicio local y habilitando su gestión desde **Prefect Cloud**: ejecución automática, logs, reintentos y monitoreo centralizado.

⚠️ Al activarla, la celda permanecerá ocupada hasta que la detengas manualmente (*Interrupt/Stop*).  
Por ese motivo, la llamada queda **comentada por defecto** para evitar ejecuciones involuntarias.

Si se habilita, el flujo puede programarse usando una expresión **cron**, definida en el propio script `etl_opensky_flow.py`.

In [6]:
import sys, os, importlib

# Asegurar acceso al módulo en src/
sys.path.append(os.path.abspath("../src"))

etl = importlib.import_module("etl_opensky_flow")

# etl.etl_opensky_flow.serve(
#     name="ETL-OpenSky",
#     cron="0 * * * *"   # Ejemplo: cada hora en punto
# )

## Monitoreo en Prefect

El monitoreo del flujo no se realiza desde el notebook, sino desde la **UI de Prefect Cloud**.
Una vez que el flow se ejecuta (ya sea por corrida única o servido como agente), es posible:

1. **Flows** → ver el listado de flows registrados (por ejemplo, `ETL-OpenSky`).  
2. **Run History** → revisar el historial de ejecuciones, con logs, tiempos y reintentos.  
3. **Blocks** → administrar credenciales, almacenamiento o infraestructura remota (si se utiliza).  
4. **Schedules** → programar ejecuciones automáticas mediante cron (requiere plan pago). 